In [4]:
import sys, importlib.metadata as m
print(sys.executable)
for p in ["huggingface_hub", "transformers", "datasets", "tokenizers", "torch", "numpy"]:
    try: print(p, m.version(p))
    except Exception: print(p, "THIẾU")

c:\Users\ADMIN\anaconda3\python.exe
huggingface_hub 1.33.0
transformers 5.17.0
datasets 5.0.1
tokenizers 0.23.2
torch 2.13.0
numpy 2.3.5


In [1]:
import os
# PHẢI đặt trước mọi import: bắt transformers chỉ dùng PyTorch, không nạp TensorFlow
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"      # tránh xung đột OpenMP trên Windows/Anaconda
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch                                      # import torch đầu tiên
print("torch ok |", torch.__version__, "| GPU:", torch.cuda.is_available())

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))        # chạy notebook từ thư mục notebooks/

import time
from datasets import Dataset
from scipy.special import softmax
from sklearn.metrics import accuracy_score
print("datasets/scipy/sklearn ok")

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed as hf_seed)
print("transformers ok")

from src.config import LABELS, MODELS, SEED, set_seed
from src.data import nap_du_lieu
from src.evaluate import do_thoi_gian_du_doan, luu_ket_qua
set_seed(); hf_seed(SEED)
print("src ok")

TEN, MAX_LEN_BERT = "distilbert-base-uncased", 128     # tiếng Việt: "vinai/phobert-base" (cần tách từ)

torch ok | 2.13.0+cpu | GPU: False
datasets/scipy/sklearn ok
transformers ok
src ok


In [2]:
data = nap_du_lieu()
tok = AutoTokenizer.from_pretrained(TEN)

def tao_dataset(k):
    ds = Dataset.from_dict({"text": list(data["texts"][k]), "label": data["y"][k].tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN_BERT),
                  batched=True, remove_columns=["text"])

ds = {k: tao_dataset(k) for k in ("train", "val", "test")}

Using the latest cached version of the dataset since ag_news couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\ADMIN\.cache\huggingface\datasets\ag_news\default\0.0.0\eb185aade064a813bc0b7f42de02595523103ca4 (last modified on Tue Sep 29 14:44:32 2026).


Map:   0%|          | 0/108000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [4]:
import math

model = AutoModelForSequenceClassification.from_pretrained(TEN, num_labels=len(LABELS))

BATCH_TRAIN, EPOCHS = 32, 3
tong_buoc = math.ceil(len(ds["train"]) / BATCH_TRAIN) * EPOCHS          # ĐỔI: tự tính tổng số bước
warmup = int(0.1 * tong_buoc)                                           # ĐỔI: 10% số bước làm warmup

args = TrainingArguments(
    output_dir=str(MODELS / "distilbert_ckpt"), learning_rate=3e-5, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_TRAIN, per_device_eval_batch_size=128, weight_decay=0.01,
    warmup_steps=warmup,                                                # ĐỔI: thay cho warmup_ratio=0.1
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="accuracy", save_total_limit=1,
    fp16=torch.cuda.is_available(), seed=SEED, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["val"],
                  data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))})
t0 = time.perf_counter()
trainer.train()
train_s = time.perf_counter() - t0

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ADMIN\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(l

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
out = trainer.predict(ds["test"])
proba = softmax(out.predictions, axis=1)

model.eval()
def du_doan_1_tin(text):
    enc = tok(text, truncation=True, max_length=MAX_LEN_BERT, return_tensors="pt").to(model.device)
    with torch.no_grad():
        model(**enc)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

luu_ket_qua("distilbert", data["y"]["test"], proba.argmax(1), train_s,
            sum(p.numel() for p in model.parameters()),
            do_thoi_gian_du_doan(du_doan_1_tin, data["texts"]["test"]), proba,
            extra={"lr": 3e-5, "epochs": 3})
trainer.save_model(str(MODELS / "distilbert_final"))